# RAG Evaluation with Llama Stack Eval API

This notebook demonstrates pre-RAG vs post-RAG evaluation using the official
Llama Stack evaluation API (`/v1alpha/eval/benchmarks` + `/v1beta/datasets`).

**Pattern:** [fantaco-redhat-one-2026/evals-llama-stack](https://github.com/burrsutter/fantaco-redhat-one-2026/tree/main/evals-llama-stack)  
**Ref:** [RHOAI 3.3 — Evaluating AI Systems](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.3/html/evaluating_ai_systems)

**Workflow:**
1. Register pre-RAG dataset (LLM answers without document context)
2. Register post-RAG dataset (LLM answers WITH retrieved context)
3. Register benchmarks with `basic::subset_of` + LLM-as-judge scoring
4. Run evaluations and compare pre vs post RAG scores

In [ ]:
import os
import json
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger('httpx').setLevel(logging.WARNING)

from llama_stack_client import LlamaStackClient, BadRequestError

LLAMASTACK_URL = os.getenv('LLAMASTACK_URL', 'http://lsd-rag-service.private-ai.svc.cluster.local:8321')
MODEL_ID = 'vllm-granite-agent/granite-8b-agent'

client = LlamaStackClient(base_url=LLAMASTACK_URL)
print(f'Connected to: {LLAMASTACK_URL}')
print(f'Models: {[m.id for m in client.models.list()]}')

## 1. Define Test Cases (Pre-RAG and Post-RAG)

Same questions for both. Pre-RAG: LLM answers from training data only.  
Post-RAG: we'll retrieve context from pgvector and include it in the prompt.

In [ ]:
import requests

# Vector store IDs
stores = client.vector_stores.list()
store_map = {vs.name: vs.id for vs in stores.data}
print(f'Vector stores: {store_map}')

# Test cases per scenario
scenarios = {
    'acme_corporate': [
        {'q': 'What are the key calibration procedures for DFO lithography?',
         'a': 'DFO calibration involves alignment verification, dose calibration, focus optimization, and overlay measurement.'},
        {'q': 'What products and standards does ACME cover?',
         'a': 'ACME covers EUV and DUV lithography products aligned with SEMI and ISO standards.'},
        {'q': 'What is the maintenance schedule for the L-900 tool?',
         'a': 'L-900 requires FMEA-based failure monitoring, vibration analysis, and temperature trending.'},
    ],
    'whoami': [
        {'q': 'What is this persons professional background?',
         'a': 'Technology professional with experience in cloud computing, Kubernetes, and AI/ML platforms.'},
    ],
}
print(f'Total test cases: {sum(len(v) for v in scenarios.values())}')

## 2. Generate Pre-RAG and Post-RAG Answers

In [ ]:
def call_llm(question, context=None):
    """Call LLM via /v1/chat/completions."""
    messages = []
    if context:
        messages.append({'role': 'system', 'content': f'Answer based on this context:\n{context[:3000]}'})
    messages.append({'role': 'user', 'content': question})
    
    resp = requests.post(
        f'{LLAMASTACK_URL}/v1/chat/completions',
        json={'model': MODEL_ID, 'messages': messages, 'max_tokens': 512, 'temperature': 0, 'stream': False},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()['choices'][0]['message']['content']

def retrieve_context(question, store_id):
    """Retrieve from pgvector via /v1/vector_stores/{id}/search."""
    resp = requests.post(
        f'{LLAMASTACK_URL}/v1/vector_stores/{store_id}/search',
        json={'query': question, 'max_num_results': 3},
        timeout=30,
    )
    resp.raise_for_status()
    chunks = []
    for r in resp.json().get('data', []):
        for c in r.get('content', []):
            chunks.append(c.get('text', ''))
    return '\n---\n'.join(chunks)

# Generate answers
pre_rag_rows = []
post_rag_rows = []

for scenario, tests in scenarios.items():
    store_id = store_map.get(scenario)
    print(f'\n=== {scenario} (store: {store_id}) ===')
    
    for t in tests:
        q, expected = t['q'], t['a']
        
        # Pre-RAG
        pre_answer = call_llm(q)
        pre_rag_rows.append({
            'input_query': q,
            'expected_answer': expected,
            'chat_completion_input': json.dumps([{'role': 'user', 'content': q}]),
        })
        print(f'  Pre-RAG: {pre_answer[:80]}...')
        
        # Post-RAG
        if store_id:
            ctx = retrieve_context(q, store_id)
            post_answer = call_llm(q, context=ctx)
        else:
            post_answer = pre_answer
        post_rag_rows.append({
            'input_query': q,
            'expected_answer': expected,
            'chat_completion_input': json.dumps([{'role': 'user', 'content': q}]),
        })
        print(f'  Post-RAG: {post_answer[:80]}...')

print(f'\nGenerated {len(pre_rag_rows)} pre-RAG and {len(post_rag_rows)} post-RAG rows')

## 3. Register Datasets

In [ ]:
for ds_id in ['pre-rag-eval', 'post-rag-eval']:
    try:
        client.beta.datasets.unregister(ds_id)
        print(f'Unregistered old: {ds_id}')
    except Exception:
        pass

client.beta.datasets.register(
    purpose='eval/question-answer',
    source={'type': 'rows', 'rows': pre_rag_rows},
    dataset_id='pre-rag-eval',
    metadata={'description': 'Pre-RAG baseline (LLM only, no document context)'},
    extra_body={'provider_id': 'localfs'},
)
print(f'Registered: pre-rag-eval ({len(pre_rag_rows)} rows)')

client.beta.datasets.register(
    purpose='eval/question-answer',
    source={'type': 'rows', 'rows': post_rag_rows},
    dataset_id='post-rag-eval',
    metadata={'description': 'Post-RAG (answers generated with retrieved document context)'},
    extra_body={'provider_id': 'localfs'},
)
print(f'Registered: post-rag-eval ({len(post_rag_rows)} rows)')

print(f'All datasets: {[d.identifier for d in client.beta.datasets.list()]}')

## 4. Register LLM-as-Judge Scoring Function

In [ ]:
JUDGE_PROMPT = """Evaluate the response quality.

Question: {input_query}
Expected: {expected_answer}
Generated: {generated_answer}

Select: (A) subset (B) superset (C) same (D) disagrees (E) insignificant diff.
Answer: """

try:
    client.scoring_functions.register(
        scoring_fn_id='rag-quality-judge',
        description='LLM-as-judge for RAG answer quality',
        return_type={'type': 'string'},
        provider_id='llm-as-judge',
        provider_scoring_fn_id='llm-as-judge-base',
        params={
            'type': 'llm_as_judge',
            'judge_model': MODEL_ID,
            'prompt_template': JUDGE_PROMPT,
        },
    )
    print('Registered: rag-quality-judge')
except BadRequestError as e:
    if 'already exists' in str(e):
        print('rag-quality-judge already exists')
    else:
        raise

print(f'Scoring functions: {[f.identifier for f in client.scoring_functions.list()]}')

## 5. Register Benchmarks

In [ ]:
for bm_id, ds_id in [('pre-rag-benchmark', 'pre-rag-eval'), ('post-rag-benchmark', 'post-rag-eval')]:
    try:
        client.alpha.benchmarks.register(
            benchmark_id=bm_id,
            dataset_id=ds_id,
            scoring_functions=['basic::subset_of', 'rag-quality-judge'],
            extra_body={'provider_id': 'meta-reference'},
        )
        print(f'Registered: {bm_id}')
    except BadRequestError as e:
        if 'already exists' in str(e):
            print(f'{bm_id} already exists')
        else:
            raise

print(f'Benchmarks: {[b.identifier for b in client.alpha.benchmarks.list()]}')

## 6. Run Evaluations

In [ ]:
eval_config = {
    'eval_candidate': {
        'type': 'model',
        'model': MODEL_ID,
        'sampling_params': {'strategy': {'type': 'greedy'}, 'max_tokens': 512},
    },
    'scoring_params': {
        'basic::subset_of': {'type': 'basic', 'aggregation_functions': ['accuracy']},
    },
}

results = {}
for bm_id in ['pre-rag-benchmark', 'post-rag-benchmark']:
    print(f'\nRunning: {bm_id}...')
    job = client.alpha.eval.run_eval(bm_id, benchmark_config=eval_config)
    print(f'  Job: {job.job_id}')
    result = client.alpha.eval.jobs.retrieve(job_id=job.job_id, benchmark_id=bm_id)
    results[bm_id] = result
    
    for fn_id, sr in (result.scores or {}).items():
        print(f'  {fn_id}: {sr.aggregated_results}')

print('\nEvaluations complete.')

## 7. Compare Pre-RAG vs Post-RAG Results

In [ ]:
print('=' * 80)
print('PRE-RAG vs POST-RAG COMPARISON')
print('=' * 80)

for mode in ['pre-rag-benchmark', 'post-rag-benchmark']:
    result = results[mode]
    label = 'PRE-RAG (baseline)' if 'pre' in mode else 'POST-RAG (with docs)'
    print(f'\n--- {label} ---')
    
    for fn_id, sr in (result.scores or {}).items():
        if sr.aggregated_results:
            print(f'  {fn_id}: {sr.aggregated_results}')
        for i, row in enumerate(sr.score_rows or []):
            gen = result.generations[i] if result.generations and i < len(result.generations) else {}
            q = gen.get('input_query', '?')
            ans = gen.get('generated_answer', '?')
            score = row.get('score', '?')
            feedback = row.get('judge_feedback', '')
            print(f'  [{i}] Q: {q[:60]}')
            print(f'       Score: {score}  {feedback[:80] if feedback else ""}')
            print(f'       Answer: {str(ans)[:100]}')

print('\n' + '=' * 80)
print('Pre-RAG shows LLM hallucinations. Post-RAG shows grounded answers from documents.')
print('=' * 80)